# Chapter 7 &mdash; Why NFA Are Succinct, and No More Powerful

**Concept 2 of the Chapter 7 decomposition:** *Why NFA Are Succinct, and Why They Are No More Powerful*

Linear in the look-back instead of exponential &mdash; yet still exactly the regular languages.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Why-NFA-Are-Succinct/Concept-Why-NFA-Are-Succinct.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Two facts that sound contradictory but are not:

* **NFA are succinct.** "The $N$-th-last symbol is a 1" needs $N{+}1$ NFA states and
  $2^N$ DFA states. The gap is genuinely exponential.
* **NFA are no more powerful.** Subset construction (Concept 8) turns any NFA into a
  DFA, so the two recognise exactly the same class: the **regular languages**.

Succinctness is about **size**, power is about **which languages**. Nondeterminism
buys you a shorter description, never a bigger class.

## 2. Definitions

### The $N$-th-last-symbol family, as NFA

In [ ]:
def nth_last_nfa(N):
    lines = ['NFA', 'I : 0 | 1 -> I', 'I : 1 -> S1']
    for k in range(1, N-1):
        lines.append('S%d : 0 | 1 -> S%d' % (k, k+1))
    if N > 1:
        lines.append('S%d : 0 | 1 -> F' % (N-1))
    else:
        lines = ['NFA', 'I : 0 | 1 -> I', 'I : 1 -> F']
    return md2mc('\n'.join(lines))

### The reference specification

In [ ]:
def spec(N): return lambda s: len(s) >= N and s[-N] == '1'

## 3. Tests

Each NFA is linear in $N$; the equivalent minimal DFA is exponential.

In [ ]:
print("%-4s %-10s %-14s" % ("N", "NFA |Q|", "min DFA |Q|"))
for N in range(1, 6):
    A = nth_last_nfa(N)
    D = min_dfa(nfa2dfa(A))
    print("%-4d %-10d %-14d  (2^N = %d)" % (N, len(A["Q"]), len(D["Q"]), 2**N))
    assert len(A["Q"]) == N + 1
    assert len(D["Q"]) >= 2**N

Same language both ways &mdash; succinctness costs nothing in expressiveness.

In [ ]:
from itertools import product
for N in range(1, 5):
    A, f = nth_last_nfa(N), spec(N)
    D = nfa2dfa(A)
    strs = [''.join(p) for k in range(9) for p in product('01', repeat=k)]
    assert all(accepts_nfa(A, s) == f(s) for s in strs)
    assert all(accepts_dfa(D, s) == f(s) for s in strs)
    print("N=%d : NFA and its subset-construction DFA both match the spec" % N)

The class of languages is unchanged &mdash; every DFA *is* already an NFA.

In [ ]:
D = md2mc('''DFA
IF : 0 -> Od
IF : 1 -> IF
Od : 0 -> IF
Od : 1 -> Od
''')
print("a DFA has one start state; an NFA has a set. That is the only change needed.")
print("DFA q0 = %r  ->  NFA Q0 = %r" % (D["q0"], {D["q0"]}))

## 4. Animation

The 4-state NFA for $N=3$ &mdash; compare it with the 8-state DFA of Chapter 5.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(nth_last_nfa(3), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Plot NFA size against minimal-DFA size for $N$ up to 8. What curve is it?
2. Is the exponential gap achievable in the *other* direction? Why not?
3. Name a decision problem that is cheap for DFA and expensive for NFA.

In [ ]:
# Your work for the exercises above.